In [1]:
import pandas as pd
import os

# 1. Verify file path and load the raw dataset
file_path = "../data/DataCoSupplyChainDataset.csv"
print(f"File exists: {os.path.exists(file_path)}")

# Load the dataset
df = pd.read_csv(file_path, encoding='latin-1')

print(f"\nDataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nFirst 5 Columns:")
print(df.columns[:5].tolist())

# Preview the first 3 rows
df.head(3)

File exists: True

Dataset Shape: 180,519 rows x 53 columns

First 5 Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer']


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [2]:
import sqlite3

# 1. Clean column names to snake_case
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('__', '_')
)

# Rename specific messy column names for clarity
rename_map = {
    'order_date_dateorders': 'order_date',
    'shipping_date_dateorders': 'shipping_date',
    'days_for_shipping_real': 'shipping_days_real',
    'days_for_shipment_scheduled': 'shipping_days_scheduled'
}
df = df.rename(columns=rename_map)

# 2. Parse dates
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['shipping_date'] = pd.to_datetime(df['shipping_date'], errors='coerce')

# 3. Create database directory and connect to SQLite
os.makedirs("../database", exist_ok=True)
db_path = "../database/supply_chain.db"
conn = sqlite3.connect(db_path)

# 4. Create normalized dim_products table
dim_products = df[[
    'product_card_id', 'product_name', 'category_id', 
    'category_name', 'department_name', 'product_price'
]].drop_duplicates(subset=['product_card_id']).copy()

dim_products.to_sql('dim_products', conn, if_exists='replace', index=False)

# 5. Create normalized dim_customers table
dim_customers = df[[
    'customer_id', 'customer_fname', 'customer_lname', 
    'customer_segment', 'customer_city', 'customer_state', 
    'customer_country', 'market', 'order_region'
]].drop_duplicates(subset=['customer_id']).copy()

dim_customers.to_sql('dim_customers', conn, if_exists='replace', index=False)

# 6. Create fact_orders table
fact_cols = [
    'order_item_id', 'order_id', 'customer_id', 'product_card_id',
    'order_date', 'shipping_date', 'shipping_mode', 'delivery_status',
    'late_delivery_risk', 'shipping_days_real', 'shipping_days_scheduled',
    'sales', 'order_item_quantity', 'order_item_discount', 
    'order_item_total', 'benefit_per_order', 'order_status', 'type'
]

fact_orders = df[fact_cols].copy()
fact_orders.to_sql('fact_orders', conn, if_exists='replace', index=False)

conn.commit()
conn.close()

print("✅ Success! Cleaned data staged into SQLite database at '../database/supply_chain.db'")
print(f" - dim_products: {len(dim_products)} rows")
print(f" - dim_customers: {len(dim_customers)} rows")
print(f" - fact_orders: {len(fact_orders)} rows")

✅ Success! Cleaned data staged into SQLite database at '../database/supply_chain.db'
 - dim_products: 118 rows
 - dim_customers: 20652 rows
 - fact_orders: 180519 rows


In [4]:
# Export the cleaned dataframe to the data/ folder for Power BI
output_path = "../data/cleaned_supply_chain.csv"
df.to_csv(output_path, index=False)

print(f"✅ Successfully exported cleaned dataset to: {output_path}")
print(f"Total rows exported: {df.shape[0]:,} rows x {df.shape[1]} columns")

✅ Successfully exported cleaned dataset to: ../data/cleaned_supply_chain.csv
Total rows exported: 180,519 rows x 53 columns


In [3]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect("../database/supply_chain.db")

# 1. Shipping Mode Performance & Late Delivery Risk Scorecard
q1 = """
SELECT 
    shipping_mode,
    COUNT(order_item_id) AS total_orders,
    SUM(late_delivery_risk) AS delayed_orders_count,
    ROUND(SUM(late_delivery_risk) * 100.0 / COUNT(order_item_id), 2) AS delay_percentage,
    ROUND(AVG(shipping_days_real - shipping_days_scheduled), 2) AS avg_delay_variance_days
FROM fact_orders
GROUP BY shipping_mode
ORDER BY delay_percentage DESC;
"""
df_shipping = pd.read_sql_query(q1, conn)
print("--- 1. Shipping Mode Performance ---")
display(df_shipping)

# 2. Top Regions by Revenue at Risk Due to Late Deliveries
q2 = """
SELECT 
    c.market,
    c.order_region,
    COUNT(f.order_item_id) AS total_orders_at_risk,
    ROUND(SUM(f.order_item_total), 2) AS total_revenue_at_risk
FROM fact_orders f
JOIN dim_customers c ON f.customer_id = c.customer_id
WHERE f.late_delivery_risk = 1
GROUP BY c.market, c.order_region
ORDER BY total_revenue_at_risk DESC
LIMIT 5;
"""
df_regions = pd.read_sql_query(q2, conn)
print("\n--- 2. Top 5 Regions by Revenue at Risk ---")
display(df_regions)

conn.close()

--- 1. Shipping Mode Performance ---


,shipping_mode,total_orders,delayed_orders_count,delay_percentage,avg_delay_variance_days
0,First Class,27814,26513,95.32,1.00
1,Second Class,35216,26987,76.63,1.99
2,Same Day,9737,4454,45.74,0.48
3,Standard Class,107752,41023,38.07,-0.00



--- 2. Top 5 Regions by Revenue at Risk ---


,market,order_region,total_orders_at_risk,total_revenue_at_risk
0,Europe,Western Europe,15867,3057463.64
1,LATAM,Central America,15221,2738982.84
2,LATAM,South America,8885,1588545.81
3,Europe,Northern Europe,5445,1050452.07
4,Europe,Southern Europe,5406,1024646.00
